# 1. Title and Project Overview
## US Traffic Accident Severity Analytics Using PySpark
### Big Data Analytics — Final Comprehensive Project Notebook

---

### Project Overview
Traffic accidents represent a critical threat to public safety, emergency service readiness, and transportation infrastructure efficiency across the United States. Analyzing accident patterns at a continental scale requires processing millions of high-dimensional records containing geographic coordinates, precise timestamps, weather observations, and roadway infrastructure indicators.

This project implements a scalable **Big Data Analytics (BDA)** pipeline utilizing **Apache Spark (PySpark)** to ingest, profile, analyze, and model over **7.7 million records** from the nationwide US Accidents dataset (covering 2016 through 2023).

### Core Objective
> *"To analyze large-scale US traffic accident data using PySpark, identify important temporal, geographic, weather and infrastructure patterns, and evaluate machine-learning models for multi-class accident severity prediction."*

### Analytical Framing
All findings presented in this notebook represent **observational associations** derived from historical event logs. In accordance with rigorous data science principles, observed correlations (e.g., peak accident counts during commuter windows or specific weather conditions) reflect reporting volumes and environmental states rather than causal relationships.

# 2. Technologies Used

The analytical and machine learning pipelines were developed using modern big data technologies optimized for local execution under hardware constraints:

- **Programming Language**: Python 3.13.2
- **Distributed Computing Engine**: Apache Spark (PySpark 4.2.0)
- **Runtime Environment**: Java 17 (OpenJDK Adoptium 17.0.20.101-hotspot)
- **Hadoop Ecosystem Integration**: Hadoop 3.5.0 client libraries with native Windows binaries (`winutils.exe`, `hadoop.dll`)
- **Query & Transformation APIs**: Spark DataFrames and Spark SQL
- **Machine Learning**: PySpark MLlib (`Pipeline`, `StringIndexer`, `OneHotEncoder`, `VectorAssembler`, `Imputer`, `LogisticRegression`, `DecisionTreeClassifier`, `RandomForestClassifier`)
- **Analytics & Visualization**: Pandas (restricted strictly to small aggregated summary tables), Matplotlib, and NumPy

### Hardware Environment & Memory-Aware Engineering
- **Host RAM**: Approximately 7.37 GB total physical memory.
- **Resource Constraints**: Processing 7.7M rows in local mode requires conservative memory governance (`spark.driver.memory=2g`, `spark.sql.shuffle.partitions=2`, `local[1]`).
- **Design Rule**: The full dataset was never collected into Python memory (`collect()` on raw data was prohibited). Parquet columnar storage with Snappy compression was utilized to minimize I/O overhead.

In [ ]:
# Setup environment and import core libraries
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Verify environment paths
PROJECT_ROOT = Path("..").resolve()
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
CHARTS_DIR = PROJECT_ROOT / "outputs" / "charts"
MODELS_DIR = PROJECT_ROOT / "outputs" / "models"
PARQUET_DIR = PROJECT_ROOT / "data" / "processed" / "us_accidents_parquet"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Parquet Path Exists: {PARQUET_DIR.exists()}")
print(f"Tables Directory Exists: {TABLES_DIR.exists()}")
print(f"Charts Directory Exists: {CHARTS_DIR.exists()}")

# 3. Dataset Description

The primary dataset is the **US Accidents Dataset (March 2023 release)**, converted from raw CSV (3,058,183,727 bytes) into optimized columnar Snappy Parquet format (`data/processed/us_accidents_parquet/`). 

### Verified Dataset Dimensions
- **Total Records (Rows)**: 7,728,394
- **Total Attributes (Columns)**: 46
- **Distinct Accident IDs**: 7,728,394 (100% unique primary key)
- **Temporal Span**: January 15, 2016 01:48:33 to April 1, 2023 05:00:00 (~7.2 years)
- **Geographic Coverage**: 49 US States (contiguous US plus District of Columbia), 13,678 Cities, 1,871 Counties.

In [ ]:
# Load and inspect dataset overview table
overview_df = pd.read_csv(TABLES_DIR / "dataset_overview.csv")
print("=== US ACCIDENTS DATASET SUMMARY ===")
display(overview_df)

### Schema and Attribute Categories
The 46 columns encompass six distinct feature groups:
1. **Identification**: `ID`, `Source`
2. **Temporal**: `Start_Time`, `End_Time`, `Weather_Timestamp`
3. **Geographic**: `Start_Lat`, `Start_Lng`, `End_Lat`, `End_Lng`, `Distance(mi)`, `Street`, `City`, `County`, `State`, `Zipcode`, `Country`, `Timezone`, `Airport_Code`
4. **Target**: `Severity` (integer values 1 to 4)
5. **Meteorological**: `Temperature(F)`, `Wind_Chill(F)`, `Humidity(%)`, `Pressure(in)`, `Visibility(mi)`, `Wind_Direction`, `Wind_Speed(mph)`, `Precipitation(in)`, `Weather_Condition`
6. **Infrastructure (Boolean)**: `Amenity`, `Bump`, `Crossing`, `Give_Way`, `Junction`, `No_Exit`, `Railway`, `Roundabout`, `Station`, `Stop`, `Traffic_Calming`, `Traffic_Signal`, `Turning_Loop`
7. **Twilight / Solar Angles**: `Sunrise_Sunset`, `Civil_Twilight`, `Nautical_Twilight`, `Astronomical_Twilight`

# 4. Data Quality and Missing Values

During Stage 3 and Stage 4 data profiling, extensive quality checks and anomaly investigations were conducted across all 7,728,394 records.

### Missingness Profile
The dataset exhibits substantial missingness, particularly in secondary sensor and coordinate fields:
- **`End_Lat` / `End_Lng`**: **44.03% missing** (3,402,773 nulls) — recorded primarily in specific feed sources.
- **`Precipitation(in)`**: **28.51% missing** (2,203,588 nulls) — many weather stations do not record precipitation unless active rain/snow occurs.
- **`Wind_Chill(F)`**: **25.87% missing** (1,999,019 nulls) — calculated only under cold and windy conditions.
- **`Wind_Speed(mph)`**: **7.39% missing** (571,233 nulls).
- **`Visibility(mi)`**: **2.29% missing** (177,098 nulls).
- **`Humidity(%)`**: **2.25% missing** (174,142 nulls).
- **`Weather_Condition`**: **2.24% missing** (173,459 nulls).
- **`Temperature(F)`**: **2.12% missing** (163,853 nulls).
- **`Pressure(in)`**: **1.82% missing** (140,679 nulls).

### Extreme Numeric Values Observed
- **Wind Speed**: Maximum recorded value of 1,087.0 mph (sensor malfunction/telemetry anomaly).
- **Atmospheric Pressure**: Range from 0.0 in to 58.63 in (standard sea-level pressure is ~29.92 in).
- **Visibility**: Extreme recorded readings up to 140 miles.
- **Constant Feature**: `Turning_Loop` is `False` across all 7,728,394 rows (0% information content).

### Methodological Handling
Rather than blindly dropping rows with missing values (which would discard over 40% of the entire dataset and induce selection bias), missingness was handled systematically:
- **Numeric Features**: Imputed using the **median** learned strictly on the training partition (robust against extreme outliers).
- **Categorical Features**: Null values coalesced into an explicit `"Unknown"` category with safe string indexing (`handleInvalid="keep"`).
- **Boolean Features**: Nulls coalesced safely to `0.0`.

In [ ]:
# Summary of Top Missing Value Features
missing_data = {
    "Feature": [
        "End_Lat / End_Lng", "Precipitation(in)", "Wind_Chill(F)", 
        "Wind_Speed(mph)", "Visibility(mi)", "Humidity(%)", 
        "Weather_Condition", "Temperature(F)", "Pressure(in)", "Turning_Loop"
    ],
    "Missing Percentage": [
        "44.03%", "28.51%", "25.87%", "7.39%", "2.29%", 
        "2.25%", "2.24%", "2.12%", "1.82%", "0.00% (Constant False)"
    ],
    "Handling Strategy in ML": [
        "Excluded (Leakage / Sparse)", "Median Imputation", "Excluded (Collinear with Temp)", 
        "Median Imputation", "Median Imputation", "Median Imputation", 
        "Coalesced to 'Unknown'", "Median Imputation", "Median Imputation", "Excluded (Zero Variance)"
    ]
}
display(pd.DataFrame(missing_data))

# 5. Severity Distribution

The target variable **`Severity`** is defined on an ordinal scale of 1 to 4:
- **Severity 1**: Minor disruption / shortest delay (~0.87%)
- **Severity 2**: Moderate disruption / short delay (~79.67% — dominant class)
- **Severity 3**: Serious traffic disruption / significant congestion (~16.81%)
- **Severity 4**: Critical event / major road closure and long delays (~2.65%)

### The Class Imbalance Problem
The dataset exhibits severe natural class imbalance. Severity 2 accounts for nearly four out of five recorded accidents. If an unweighted machine learning classifier is applied, it can achieve ~79.7% raw accuracy simply by always predicting Severity 2, while completely failing to detect rare and critical Severity 1 and Severity 4 events.

To address this, the ML pipeline avoids collapsing the problem into binary classification and instead implements **balanced class weighting** within PySpark MLlib.

In [ ]:
# Load and display verified Severity Distribution
sev_dist_df = pd.read_csv(TABLES_DIR / "severity_distribution.csv")
display(sev_dist_df)

# Display Severity Distribution Chart
display(Image(filename=str(CHARTS_DIR / "severity_distribution.png")))

# 6. Temporal Analytics

Temporal analysis reveals how accident occurrences align with human activity cycles, work schedules, and seasonal factors.

### Verified Findings
- **Day of Week**: **76.55% of all accidents occur on weekdays** (Monday through Friday).
  - Peak Day: **Friday** with **1,237,229 records (16.01%)**.
  - Weekend records account for only 23.45% combined.
- **Hourly Concentration**:
  - Morning Peak: **7:00 AM – 8:00 AM** accounts for **14.09%** of daily accidents.
  - Evening Peak: **4:00 PM – 5:00 PM** accounts for **13.41%** of daily accidents.
  - Overnight Trough: **3:00 AM** represents the lowest point with only **0.96%** of recorded accidents.
- **Annual Trajectory**: Accident record volume increased substantially from 2016 through 2021, reflecting expanded API integrations and reporting sensor density.

In [ ]:
# Display Day of Week, Hourly, and Yearly Charts
display(Image(filename=str(CHARTS_DIR / "accidents_by_day_of_week.png")))
display(Image(filename=str(CHARTS_DIR / "accidents_by_hour.png")))
display(Image(filename=str(CHARTS_DIR / "accidents_by_year.png")))

# 7. Geographic Analytics

Spatial patterns indicate heavy geographic concentration across the contiguous United States.

### Top 5 States by Volume
1. **California (CA)**: 1,741,433 records (22.53%)
2. **Florida (FL)**: 880,192 records (11.39%)
3. **Texas (TX)**: 582,837 records (7.54%)
4. **South Carolina (SC)**: 382,557 records (4.95%)
5. **New York (NY)**: 347,960 records (4.50%)

The top three states alone (**CA, FL, TX**) account for **41.46%** of all recorded incidents nationwide.

> **Crucial Methodological Caveat**:
> *"These figures reflect raw accident-record volumes, not population-normalized or VMT-normalized accident rates. High accident volume in California, Florida, and Texas is driven by large population sizes, extensive highway networks, and dense sensor coverage. A state cannot be labeled 'more dangerous' based solely on unadjusted counts."*

In [ ]:
# Load and display Top 10 States
top_states_df = pd.read_csv(TABLES_DIR / "top_10_states.csv")
display(top_states_df.head(10))

# Display Top States and Cities Charts
display(Image(filename=str(CHARTS_DIR / "top_10_states.png")))
display(Image(filename=str(CHARTS_DIR / "top_20_cities.png")))

# 8. Weather Analytics

Meteorological conditions were analyzed to assess ambient environmental states during accident occurrences.

### Verified Findings
- **Common Conditions**: Over **81%** of all recorded accidents occur during standard, non-severe weather conditions:
  - *Fair*: 2,538,154 records (32.84%)
  - *Mostly Cloudy*: 1,013,153 records (13.11%)
  - *Clear*: 876,626 records (11.34%)
  - *Partly Cloudy*: 698,972 records (9.04%)
  - *Cloudy*: 648,800 records (8.40%)
- **Precipitation & Adverse Weather**: Light Rain accounts for 350,944 records (4.54%), followed by Rain (171,027 records) and Fog (99,238 records).
- **Interpretation**: While severe weather undeniably degrades road friction and visibility, the overwhelming majority of collisions occur in benign conditions simply because drivers travel far more total miles under clear/fair skies.

In [ ]:
# Load and display Top Weather Conditions
top_weather_df = pd.read_csv(TABLES_DIR / "top_weather_conditions.csv")
display(top_weather_df.head(10))

display(Image(filename=str(CHARTS_DIR / "top_weather_conditions.png")))

# 9. Road and Infrastructure Analytics

The dataset includes 12 binary infrastructure indicators denoting nearby physical traffic controls or roadway configurations.

### Verified Findings
- **Most Common Indicators**:
  - `Traffic_Signal`: Present in **14.80%** of records (1,143,485 incidents).
  - `Crossing`: Present in **11.31%** of records (874,019 incidents).
  - `Junction`: Present in **7.28%** of records (562,818 incidents).
  - `Station`: Present in **2.67%** of records (206,009 incidents).
  - `Stop`: Present in **2.65%** of records (204,496 incidents).
- **Severity Association**:
  - Incidents occurring at `Junction=True` exhibit a noticeably higher proportion of **Severity 3 and 4** accidents compared to surface-street controls like traffic signals or stops.
  - *Observational Interpretation*: Highway interchanges and junction merge points involve higher differential speeds and complex merging maneuvers, which correlate with higher-severity outcomes when collisions occur.

In [ ]:
# Load and display Infrastructure Summary
infra_df = pd.read_csv(TABLES_DIR / "infrastructure_features.csv")
display(infra_df)

display(Image(filename=str(CHARTS_DIR / "infrastructure_features.png")))

# 10. Distance Analytics

The `Distance(mi)` attribute measures the linear length of the road extent impacted by the traffic accident.

### Verified Findings
- **Point Incidents**: **42.73%** of all accident records have `Distance(mi) = 0.0` (indicating localized impact without extensive roadway queue length).
- **Distance by Severity**:
  - Severity 1: Mean distance ~0.44 miles, median 0.0 miles.
  - Severity 2: Mean distance ~0.30 miles, median 0.0 miles.
  - Severity 3: Mean distance ~0.76 miles, median 0.16 miles.
  - Severity 4: Mean distance **1.50 miles**, median **0.47 miles**.
- Higher severity events correlate strongly with larger physical corridor blockages and longer queue disruptions.

In [ ]:
# Load and display Distance Summary
dist_summary_df = pd.read_csv(TABLES_DIR / "distance_summary.csv")
dist_buckets_df = pd.read_csv(TABLES_DIR / "distance_buckets.csv")
display(dist_summary_df)
display(dist_buckets_df)

display(Image(filename=str(CHARTS_DIR / "distance_buckets.png")))

# 11. Spark SQL Demonstration

Apache Spark SQL provides an expressive declarative interface for distributed big data processing. In Stage 6, five core queries were executed to aggregate accident records across multiple dimensions.

### Query 1: Accident Count by State
```sql
SELECT State, COUNT(*) AS Accident_Count
FROM us_accidents
GROUP BY State
ORDER BY Accident_Count DESC;
```

### Query 2: Severity Distribution and Percentage
```sql
SELECT Severity, COUNT(*) AS Accident_Count,
       ROUND(COUNT(*) * 100.0 / 7728394, 2) AS Percentage
FROM us_accidents
GROUP BY Severity
ORDER BY Severity;
```

### Query 3: Top 10 Cities
```sql
SELECT City, State, COUNT(*) AS Accident_Count
FROM us_accidents
WHERE City IS NOT NULL
GROUP BY City, State
ORDER BY Accident_Count DESC
LIMIT 10;
```

### Query 4: Accident Count by Year
```sql
SELECT YEAR(Start_Time) AS Accident_Year, COUNT(*) AS Accident_Count
FROM us_accidents
WHERE Start_Time IS NOT NULL
GROUP BY YEAR(Start_Time)
ORDER BY Accident_Year;
```

### Query 5: Severity Distribution Across Top States
```sql
SELECT State, Severity, COUNT(*) AS Incident_Count
FROM us_accidents
WHERE State IN ('CA', 'FL', 'TX', 'SC', 'NY')
GROUP BY State, Severity
ORDER BY State, Severity;
```

In [ ]:
# Display Spark SQL Results (loaded from verified CSV outputs)
print("--- Spark SQL Query 2: Severity Distribution ---")
display(pd.read_csv(TABLES_DIR / "sql_severity_summary.csv"))

print("--- Spark SQL Query 3: Top Cities ---")
display(pd.read_csv(TABLES_DIR / "sql_top_cities.csv").head(10))

print("--- Spark SQL Query 4: Accident Count by Year ---")
display(pd.read_csv(TABLES_DIR / "sql_year_summary.csv"))

# 12. Machine Learning Problem Definition

### Multiclass Classification Formulation
The goal is to predict accident `Severity` as a **4-class classification problem**:
$$\mathcal{Y} \in \{1, 2, 3, 4\} \longrightarrow \{0.0, 1.0, 2.0, 3.0\}$$

- The target was **not** collapsed into binary classification; all four severity tiers were preserved.
- **Partitioning**: 80% Train, 20% Test split using fixed random seed `42`.
  - **Training Partition**: 6,182,916 records
  - **Testing Partition**: 1,545,478 records

# 13. Feature Selection

To guarantee defensible predictive validity and avoid target leakage, features were chosen strictly based on what is observable at incident onset:

### 32 Features Selected
- **Numeric (9)**: `Start_Lat`, `Start_Lng`, `Distance(mi)`, `Temperature(F)`, `Humidity(%)`, `Pressure(in)`, `Visibility(mi)`, `Wind_Speed(mph)`, `Precipitation(in)`
- **Categorical (7)**: `State`, `Timezone`, `Weather_Condition`, `Sunrise_Sunset`, `Civil_Twilight`, `Nautical_Twilight`, `Astronomical_Twilight`
- **Infrastructure Boolean (12)**: `Amenity`, `Bump`, `Crossing`, `Give_Way`, `Junction`, `No_Exit`, `Railway`, `Roundabout`, `Station`, `Stop`, `Traffic_Calming`, `Traffic_Signal`
- **Derived Temporal (4)**: `Year`, `Month`, `DayOfWeek`, `Hour`

### 16 Excluded Features & Leakage Prevention
1. `Severity`: Target variable itself.
2. `ID`, `Description`: Unique identifiers and unstructured narrative.
3. `Street`, `City`, `County`, `Zipcode`, `Airport_Code`: High-cardinality nominal proxies prone to memorization/overfitting.
4. `End_Time`: Encodes accident duration, which is unavailable at the onset of an incident.
5. `End_Lat`, `End_Lng`: Suffer from ~44% missingness.
6. `Weather_Timestamp`: Redundant with incident start time.
7. `Turning_Loop`: Constant `False` (zero variance).
8. `Source`, `Country`: Feed provenance and single-country constant (`"US"`).
9. `Start_Time`: Dropped after decomposing into cyclical components.

# 14. ML Preprocessing Pipeline

The preprocessing workflow was packaged into a modular PySpark ML `Pipeline`:
1. **Numeric Imputation**: `Imputer(strategy="median")` fitted strictly on the training partition.
2. **Categorical Indexing**: `StringIndexer(handleInvalid="keep")` converting strings into index integers.
3. **One-Hot Encoding**: `OneHotEncoder(handleInvalid="keep")` transforming indices into sparse vectors.
4. **Vector Assembly**: `VectorAssembler(handleInvalid="keep")` merging all processed features into a single unified `features` vector.

Strict leakage protection ensured that all transformers and encoders were learned **exclusively on the training split**.

# 15. Class Imbalance Handling

To overcome the 79.67% dominance of Severity 2 without resorting to computationally prohibitive SMOTE (which causes memory exhaustion on 7.7M rows), **balanced class weighting** was implemented directly in the PySpark MLlib optimization objective.

### Class Weight Formula
For each class $c \in \{1, 2, 3, 4\}$:
$$w_c = \frac{N_{\text{train}}}{4 \times N_{\text{train}, c}}$$

### Calculated Training Weights (Seed 42)
- **Severity 1** ($N = 54,009$): **$w_1 = 28.6198$**
- **Severity 2** ($N = 4,925,105$): **$w_2 = 0.3138$**
- **Severity 3** ($N = 1,039,766$): **$w_3 = 1.4866$**
- **Severity 4** ($N = 164,036$): **$w_4 = 9.4231$**

These weights penalize misclassifications of minority classes (Severity 1 and 4) proportionally higher during gradient updates.

In [ ]:
# Display Train vs Test Class Distribution Chart
display(Image(filename=str(CHARTS_DIR / "train_test_class_distribution.png")))

# 16. Models

Three supervised classifiers from PySpark MLlib were trained sequentially under memory controls:

1. **Multinomial Logistic Regression**:
   - `maxIter=20`, `regParam=0.01`, `family="multinomial"`, `weightCol="class_weight"`
   - Completed in 324.69 s (~5.4 min)
2. **Decision Tree Classifier**:
   - `maxDepth=8`, `minInstancesPerNode=20`, `weightCol="class_weight"`
   - Completed in 940.27 s (~15.7 min)
3. **Random Forest Classifier**:
   - `numTrees=25`, `maxDepth=8`, `minInstancesPerNode=20`, `seed=42`, `weightCol="class_weight"`
   - Completed in 1698.93 s (~28.3 min)

All trained models were saved to `outputs/models/` for downstream persistence.

# 17. Model Results

Model performance was evaluated on the independent **1,545,478-record test partition**. Because of extreme class imbalance, evaluation relies on **Macro F1**, **Weighted F1**, and **Weighted Precision/Recall** rather than raw accuracy alone.

### Summary Comparison Table

In [ ]:
# Load and display model comparison results
model_comp_df = pd.read_csv(TABLES_DIR / "model_comparison.csv")
display(model_comp_df)

# Display Model Comparison Metrics Chart
display(Image(filename=str(CHARTS_DIR / "model_comparison_metrics.png")))

### Key Comparative Insights
- **Decision Tree** achieved the strongest overall performance:
  - **Accuracy**: **0.4738**
  - **Weighted Precision**: **0.8378**
  - **Weighted F1**: **0.5462**
  - **Macro F1**: **0.3656**
- **Logistic Regression**: Accuracy of 0.4208, Weighted F1 of 0.5082, Macro F1 of 0.2930.
- **Random Forest**: Accuracy of 0.3869, Weighted F1 of 0.4628, Macro F1 of 0.2934.

# 18. Per-Class Evaluation

Examining class-specific metrics reveals the true operational effect of balanced class weighting.

In [ ]:
# Load and display Per-Class Evaluation Metrics
per_class_df = pd.read_csv(TABLES_DIR / "per_class_metrics.csv")
display(per_class_df)

### High Minority Sensitivity Under Balanced Weighting
- **Severity 1 (Minor Disruptions)**:
  - Decision Tree achieved **93.57% Recall** ($12,498$ correctly identified out of $13,357$).
  - Random Forest achieved **88.73% Recall**.
  - Logistic Regression achieved **80.50% Recall**.
- **Severity 4 (Critical Closures)**:
  - Decision Tree achieved **86.85% Recall** ($35,325$ correctly identified out of $40,674$).
  - Random Forest achieved **85.37% Recall**.
  - Logistic Regression achieved **55.08% Recall**.

### Precision-Recall Tradeoff
Because the models were incentivized by high minority weights ($w_1 = 28.62$, $w_4 = 9.42$) to never miss rare accidents, precision on minority classes is modest (~7% to 14%), and overall accuracy decreased from the naive 79.7% baseline to 47.4%. However, for emergency dispatch and incident response, detecting nearly 87% of critical road closures is substantially more valuable than predicting only the majority class.

# 19. Confusion Matrices

A confusion matrix visualizes how true accident severity classes map against model predictions across the 1,545,478 test instances. The diagonal elements represent correct classifications, while off-diagonal elements show specific misclassification tendencies.

In [ ]:
# Display Confusion Matrices for all three models
print("Confusion Matrix: Logistic Regression")
display(Image(filename=str(CHARTS_DIR / "confusion_matrix_logistic_regression.png")))

print("Confusion Matrix: Decision Tree Classifier")
display(Image(filename=str(CHARTS_DIR / "confusion_matrix_decision_tree.png")))

print("Confusion Matrix: Random Forest Classifier")
display(Image(filename=str(CHARTS_DIR / "confusion_matrix_random_forest.png")))

# 20. Key Findings

1. **Extreme Class Imbalance**: Severity 2 comprises **79.67%** of the 7.7M records, while Severity 1 (0.87%) and Severity 4 (2.65%) represent small minority classes.
2. **Weekday Concentration**: **76.55%** of accidents occur on weekdays, peaking on Friday (**16.01%**).
3. **Diurnal Bimodal Peaks**: Accidents concentrate heavily around morning (**7–8 AM: 14.09%**) and evening (**4–5 PM: 13.41%**) commuter windows.
4. **Geographic Clustering**: Three states (**CA, FL, TX**) account for **41.46%** of all recorded incidents nationwide.
5. **Prevalence of Fair Weather**: Over **81%** of accidents occur during fair or cloudy weather conditions, reflecting baseline exposure patterns.
6. **Infrastructure Associations**: Records marked with `Junction=True` demonstrate an elevated proportion of higher-severity (Severity 3 and 4) events.
7. **Spatial Impact Correlation**: Severe accidents (Severity 4) impact an average road extent of **1.50 miles** (median 0.47 mi), compared to ~0.30 miles for Severity 2.
8. **Class Weighting Efficacy**: Balanced class weights successfully boosted minority class recall to **93.57%** (Severity 1) and **86.85%** (Severity 4) in the Decision Tree model.
9. **Algorithm Performance**: The Decision Tree classifier demonstrated superior overall balance, leading across Accuracy (0.4738), Weighted F1 (0.5462), and Macro F1 (0.3656).
10. **Scalable Big Data Execution**: Apache Spark successfully ingested, cleaned, engineered features, and trained ML pipelines on 7.7M rows within a constrained local memory budget (~7.37 GB RAM).

# 21. Limitations

1. **Observational Nature**: The dataset comprises observational reporting records; observed correlations cannot prove causation.
2. **Lack of Exposure Denominators**: Counts are not normalized by traffic volume or Vehicle Miles Traveled (VMT), precluding risk-rate conclusions.
3. **Severe Class Imbalance**: The ~80% dominance of Severity 2 requires aggressive loss re-weighting, which inherently lowers precision on minority classes.
4. **Sensor & Meteorological Missingness**: High missingness in fields like precipitation (28.5%) and coordinates (44.0%) necessitated imputation and feature exclusion.
5. **Telemetry Outliers**: Physical anomalies (e.g., wind speed > 1,000 mph) reflect real-world sensor limitations.
6. **Hardware Constraints**: Models were trained on a single-node configuration (`local[1]`, 2 GB driver memory), bounding tree depth and ensemble size.
7. **No Real-World Deployment Claims**: Performance reflects historical test set evaluation and cannot guarantee live operational accuracy.

# 22. Conclusion

This project successfully demonstrates the application of **Apache Spark (PySpark)** for end-to-end Big Data Analytics on a massive real-world dataset of **7,728,394 US traffic accident records**.

### Summary of Achievements
- Successfully migrated 3.06 GB of raw CSV data into high-performance Snappy Parquet format.
- Conducted deep profiling, anomaly detection, temporal, geographic, weather, and infrastructure analytics using Spark DataFrames and Spark SQL.
- Formulated an operationally valid 32-feature predictor set with strict leakage prevention.
- Implemented an automated PySpark ML pipeline featuring median imputation, string indexing, one-hot encoding, and balanced class loss weighting.
- Sequentially trained and compared three MLlib classifiers (Logistic Regression, Decision Tree, Random Forest) on 6.18 million training rows and evaluated them on 1.55 million test rows.
- The **Decision Tree Classifier** emerged as the strongest performer (Accuracy: 0.4738, Weighted F1: 0.5462, Macro F1: 0.3656), successfully capturing **86.85% of critical Severity 4 accidents** and **93.57% of Severity 1 incidents**.

These results highlight both the power of distributed computing for large-scale transportation analytics and the essential role of balanced loss functions when modeling extreme real-world class imbalances.